<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

In [1]:
# === IMPORT LIBRARIES ===
import os
import sys
import subprocess
import ctypes

# === CUDA SYSTEM BOOT FIX ===

# Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# === FILEPATH SETUP ===

# 1. Inject Codebase into Python Path (Wipe cache first for Jupyter safety)
import sys
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)
print(f"✅ Codebase mounted at: {codebase_path}")

# 2. Configure Global Filepaths
CACHE_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"

✅ Codebase mounted at: ../


In [3]:
# === IMPORT HF API KEY ===
from huggingface_hub import login

# Load HF token from artifacts/.env
env_path = ENV_PATH
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            if line.strip() and not line.startswith("#") and "=" in line:
                key, val = line.strip().split("=", 1)
                if key.strip() == "HF_TOKEN":
                    login(token=val.strip())
                    print("Successfully logged into Hugging Face Hub!")
                    break
else:
    print(f"Warning: {env_path} not found.")

Successfully logged into Hugging Face Hub!


In [4]:
# === CONFIGURATION ===
USE_UNSLOTH = False  # Use Unsloth for faster training

# Qwen3-30B-A3B: text-only MoE, ~30B total / ~3B active params
# Fits on RTX 4090 (24GB) at 4-bit quantization (16.74 GB base + 3.4 GB LoRA)
MODEL_ID = "ibm-granite/granite-3.3-8b-instruct"

STRUCTONLY_OUTPUT_DIR = f"{MODELS_DIR}/granite-3.3-8b-instruct_LoRA"
FULLINFO_OUTPUT_DIR   = f"{MODELS_DIR}/granite-3.3-8b-instruct_LoRA"


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
# === IMPORT LIBRARIES ===
# IMPORTANT: Unsloth must be imported FIRST to apply all kernel patches
import os
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from transformers import AutoTokenizer, EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer
from src.utils.prompts import format_prompt


ModuleNotFoundError: No module named 'unsloth'

In [ ]:
# === LOAD TOKENIZER & PREPARE SCHEMAS ===
# Load tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

COMPUTE_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")

# Apply chat template formatting
import json as _json
def apply_chat_template(example, tokenizer):
    example = _json.loads(example["text"])
    messages = format_prompt(example)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": prompt}


In [ ]:
# === LoRA CONFIG ===
if USE_UNSLOTH == False:
	peft_config = LoraConfig(
		r=16,
		lora_alpha=32,
		lora_dropout=0.05,
		bias="none",
		task_type="CAUSAL_LM",
		target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
		# === TRYING OTHER SFT MTHODS (DoRA, PiSSA) ===
		# use_dora=True,
		# init_lora_weights="pissa"  # or just "pissa_niter_16"
	)
# else USE_UNSLOTH == True:
# LoraConfig removed because Unsloth uses FastLanguageModel.get_peft_model()
# DoRA, rsDoRA is also supported. See #3 model = FastLanguageModel.get_peft_model() below for modifications.


In [ ]:
# === MULTI-CONFIG SFT TRAINING LOOP ===
import gc
from transformers.trainer_utils import get_last_checkpoint

max_seq_length = 1500

configs = [
    {
        "prompt_format": "structOnly",
        "train_path": f"{CACHE_DIR}/train_structural.jsonl",
        "val_path":   f"{CACHE_DIR}/val_structural.jsonl",
        "output_dir": STRUCTONLY_OUTPUT_DIR
    },
    {
        "prompt_format": "fullInfo",
        "train_path": f"{CACHE_DIR}/train_full_info.jsonl",
        "val_path":   f"{CACHE_DIR}/val_full_info.jsonl",
        "output_dir": FULLINFO_OUTPUT_DIR
    }
]

use_bf16 = is_bfloat16_supported()
use_fp16 = not use_bf16

for config in configs:
    format_name = config["prompt_format"]
    print(f"\n================ STARTING TRAINING FOR {format_name} ==================")

    # 1. Load and process datasets
    print(f"Loading dataset from {config['train_path']}...")
    dataset = load_dataset("text", data_files={
        "train": config["train_path"],
        "val":   config["val_path"]
    })
    processed_dataset = dataset.map(lambda x: apply_chat_template(x, tokenizer))

    # 2. Load model with Unsloth (4-bit NF4, bfloat16 compute)
    print(f"Loading {MODEL_ID} with Unsloth 4-bit...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=max_seq_length,
        dtype=COMPUTE_DTYPE,
        load_in_4bit=False,
    )

    # 3. Attach LoRA adapters via Unsloth
    model = FastLanguageModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    model.print_trainable_parameters()

    # 4. SFT Configuration
    training_args = SFTConfig(
        output_dir=config["output_dir"],
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        logging_dir=f"{config['output_dir']}/logs",
        num_train_epochs=1,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=use_bf16,
        fp16=use_fp16,
        optim="paged_adamw_8bit",
        dataset_text_field="text",
        max_seq_length=max_seq_length,
    )

    # 5. Initialize SFTTrainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=processed_dataset["train"],
        eval_dataset=processed_dataset["val"],
        processing_class=tokenizer,
        args=training_args,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    # 6. Train (resume if checkpoint exists)
    last_checkpoint = get_last_checkpoint(config["output_dir"])
    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        trainer.train()
    trainer.save_model(config["output_dir"])

    # 7. Memory cleanup
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"🧹 Cleared GPU cache after {format_name}.")
    print("=================================================================\n")
